# Corpus similarity (embeddings)

Loads [`output/corpus_20_resumes.csv`](output/corpus_20_resumes.csv), encodes **each row** in **Summary**, **Experience**, and **Skills** with `all-MiniLM-L6-v2`, adds an **`embedding`** column (JSON array per row), and writes **`output/corpus_with_embedding.csv`**.

Set **`TARGET_RESUME_ID`** below to choose which resume is “me” for the optional ranking cell.

In [1]:
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd

_cwd = Path.cwd().resolve()
if (_cwd / "corpus_similarity.ipynb").exists():
    NOTEBOOK_DIR = _cwd
elif (_cwd / "peopleLikeMe" / "corpus_similarity.ipynb").exists():
    NOTEBOOK_DIR = _cwd / "peopleLikeMe"
else:
    NOTEBOOK_DIR = _cwd

OUTPUT_DIR = NOTEBOOK_DIR / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CORPUS_PATH = OUTPUT_DIR / "corpus_20_resumes.csv"
META_PATH = OUTPUT_DIR / "resume_metadata.csv"
OUT_EMB_PATH = OUTPUT_DIR / "corpus_with_embedding.csv"

TARGET_RESUME_ID = 1  # change to any ResumeID in the corpus

df = pd.read_csv(CORPUS_PATH, dtype={"ResumeID": int, "ItemID": int})
df["Title"] = df["Title"].fillna("").astype(str)

CORPUS_PATH.resolve(), len(df), sorted(df["ResumeID"].unique())[:5], "...", df["Section"].unique().tolist()

(WindowsPath('C:/Users/DeloresMincarelli/Documents/Portfolio/peopleLikeMe/output/corpus_20_resumes.csv'),
 236,
 [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)],
 '...',
 ['Summary', 'Experience', 'Education', 'Skills'])

### Install sentence-transformers Scikit-learn

sentence-transformers
- A library for turning sentences or short texts into fixed-size vectors (embeddings) so you can compare meaning, not just words.
- Built on PyTorch and Hugging Face Transformers.
Pre-trained models (e.g. MiniLM, MPNet) map text to dense vectors; similar meanings tend to sit close in vector space.
Common uses: semantic search, duplicate detection, clustering, re-ranking, similarity between documents or queries.
It’s the usual choice when you want “what does this sentence mean?” style similarity without training a big model from scratch.

scikit-learn

- The main Python library for classical / tabular machine learning and scientific computing around models.

Together: sentence-transformers gives you text embeddings; scikit-learn can use those vectors (or other features) for clustering, classification, nearest neighbors, etc.

### Load embedding model
all-MiniLM-L6-v2 (from SentenceTransformers) because it’s:

- Built specifically for semantic similarity
It converts text → vectors (embeddings)
Similar meaning → vectors close together (cosine similarity)
That’s exactly what your workflow needs:
Resume bullets ↔ skills/tasks matching
- Very efficient
Small model (~80MB)
Fast on CPU (no GPU needed)
Great for batching thousands of rows (like O*NET)
- “Good enough” accuracy

Is it free?
- Yes — completely free
Open-source (Apache 2.0 license)
No API calls
Runs locally
No usage limits
No cost per embedding

- That’s why it’s commonly used in: prototypes and offline pipelines

In [2]:
# pip install sentence-transformers scikit-learn

from sentence_transformers import SentenceTransformer

MODEL_NAME = "all-MiniLM-L6-v2"

model = SentenceTransformer(MODEL_NAME)

print("Model loaded successfully")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully


### Encode Summary, Experience, and Skills (one embedding per row)

Education rows are kept in the export with an **empty** `embedding`. Vectors are stored as **JSON arrays** in the CSV.

In [3]:
ENCODE_SECTIONS = {"Summary", "Experience", "Skills"}


def row_to_encode_text(row: pd.Series) -> str:
    section = str(row["Section"]).strip()
    text = str(row["Text"]).strip()
    title = str(row["Title"]).strip() if pd.notna(row["Title"]) else ""
    if section == "Summary":
        return text
    if section == "Experience":
        return f"{title}: {text}" if title else text
    if section == "Skills":
        return text
    return text


mask = df["Section"].isin(ENCODE_SECTIONS)
encode_idx = df.index[mask]
texts = [row_to_encode_text(df.loc[i]) for i in encode_idx]

vectors = model.encode(texts, show_progress_bar=True)

df_out = df.copy()
df_out["embedding"] = ""
for row_i, idx in enumerate(encode_idx):
    df_out.at[idx, "embedding"] = json.dumps(vectors[row_i].tolist())

df_out.to_csv(OUT_EMB_PATH, index=False, encoding="utf-8", na_rep="")

encoded_rows = int(mask.sum())
print("Wrote", OUT_EMB_PATH.resolve())
print("Encoded rows:", encoded_rows, "vector dim:", vectors.shape[1] if len(vectors) else 0)

Batches:   0%|          | 0/7 [00:00<?, ?it/s]

Wrote C:\Users\DeloresMincarelli\Documents\Portfolio\peopleLikeMe\output\corpus_with_embedding.csv
Encoded rows: 196 vector dim: 384


### From ChatGPT: How to compare?
Here is a concise takeaway from that conversation, without touching your project.

Core idea
A resume is a structured portfolio of meanings (summary + many bullets), not one document and not interchangeable snippets. “Professionals like me” depends on what you mean by similar (overall identity vs day-to-day work vs strongest specialties vs breadth), and no single number captures all of that.

What to avoid (as stated in the chat)

Treating “embed the embeddings” as a magic second step: if you collapse many vectors to one, that should be an explicit aggregation (mean, weighted mean, role/theme structure), not an opaque second embedding model.
One max bullet match or one plain average over all bullets as the whole story: fragile, length-sensitive, and easy to critique.
Letting summary similarity dominate: often generic and polished.
What the chat recommends instead

Multi-level comparison: summary (broad positioning), weighted bullet centroid (holistic work signal), and bullet-level alignment (concrete overlap), then a weighted composite (they suggested something like 20% / 30% / 50%).
For bullets: bidirectional aggregation (A→B and B→A) so broad vs narrow profiles are not confused.
Use top‑k (or proportional k) over per-bullet best matches so a few strong lines and noise from long resumes do not dominate a naive full average.
For scrutiny: add explainability (component scores + a few supported bullet pairs), and optionally tighten matching so one generic bullet cannot “explain” many lines (one-to-one / assignment-style matching as a later upgrade).
Bottom line
The defensible story is: similarity is estimated at several levels, aggregated with clear rules, and reported with evidence—not “one cosine score proves sameness.” That matches how embeddings are usually used in high-stakes settings: transparent signals, explicit limits on what the score means.

If you want to discuss next, we could pick one definition of “like me” (e.g. work content vs career identity) and see how that changes weights and which sections you trust most (summary vs experience vs skills).

### Step 1: Summary-level similarity (broad positioning)

**SummarySim** between two resumes is the **cosine similarity** of their **Summary** row embeddings (already stored in `output/corpus_with_embedding.csv`). This cell reads that CSV only (no re-encode), uses `sklearn.metrics.pairwise.cosine_similarity`, ranks every resume against `TARGET_RESUME_ID`, and merges `resume_metadata.csv` for context.

Summary text is often polished and uses generic phrases (*results-driven*, *cross-functional*, *data-driven*), so cosine similarity here can **overstate** how alike two people are in real work. Treat this as **one component** of a future composite score, not a full “professionals like me” verdict.

**Checks:** similarity of the target to itself should be **1.0** (within float noise). All similarities lie in **[-1, 1]**.

In [4]:
from sklearn.metrics.pairwise import cosine_similarity

if not OUT_EMB_PATH.is_file():
    raise FileNotFoundError(f"Embedded corpus not found: {OUT_EMB_PATH.resolve()}")
if not META_PATH.is_file():
    raise FileNotFoundError(f"Resume metadata not found: {META_PATH.resolve()}")

emb_df = pd.read_csv(OUT_EMB_PATH, dtype={"ResumeID": int, "ItemID": int})
sum_mask = emb_df["Section"].astype(str).str.strip() == "Summary"
sum_df = emb_df.loc[sum_mask & emb_df["embedding"].astype(str).str.len().gt(0)].copy()

counts = sum_df.groupby("ResumeID").size()
all_ids = sorted(emb_df["ResumeID"].unique())
missing_summary = [rid for rid in all_ids if rid not in counts.index or counts.loc[rid] == 0]
if missing_summary:
    raise ValueError(f"No non-empty Summary embedding for ResumeID(s): {missing_summary}")
multi = counts[counts > 1]
if len(multi):
    raise ValueError(
        "Expected exactly one Summary row per ResumeID; got duplicates: "
        f"{multi.to_dict()}"
    )

ids = sorted(sum_df["ResumeID"].unique())
if TARGET_RESUME_ID not in ids:
    raise ValueError(f"TARGET_RESUME_ID={TARGET_RESUME_ID} not in corpus summary rows")

def summary_vec(rid: int) -> np.ndarray:
    row = sum_df.loc[sum_df["ResumeID"] == rid, "embedding"].iloc[0]
    return np.array(json.loads(row), dtype=np.float32)

vectors = np.stack([summary_vec(rid) for rid in ids], axis=0)
target_row = ids.index(TARGET_RESUME_ID)
target_vec = vectors[target_row].reshape(1, -1)

sims = cosine_similarity(target_vec, vectors)[0]
rank_df = (
    pd.DataFrame({"ResumeID": ids, "summary_cosine_similarity": sims.astype(float)})
    .sort_values("summary_cosine_similarity", ascending=False)
    .reset_index(drop=True)
)
meta = pd.read_csv(META_PATH)
rank_df = rank_df.merge(meta, on="ResumeID", how="left")

self_sim = float(sims[target_row])
assert abs(self_sim - 1.0) < 1e-5, f"expected target self-similarity ~1, got {self_sim}"
assert sims.min() >= -1.0001 and sims.max() <= 1.0001

rank_df

,ResumeID,summary_cosine_similarity,primary_domain,role_family,seniority_band
0,1,1.000000,data_analytics,healthcare_analytics,mid
1,4,0.515133,data_analytics,healthcare_reporting,early
2,7,0.499845,data_science,research_ds,senior
3,8,0.444134,data_science,nlp_search,mid
4,2,0.406697,data_analytics,bi_reporting,mid
5,9,0.361967,data_science,forecasting,mid
6,14,0.335894,finance,fpna,senior
7,3,0.330743,data_analytics,operations_analytics,senior
8,13,0.329266,hr,total_rewards,mid
9,17,0.265207,engineering,structural,mid


### Step 2: Weighted experience centroid (holistic work signal)
Resume A has 10 bullets, B has 3 — what’s the impact?
Per resume: A’s centroid uses only A’s 10 vectors; B’s uses only B’s 3. Bullet counts do not get mixed when building each centroid.
Still comparable: Both centroids live in the same embedding space, so cosine between C_A and C_B is well-defined.
How count matters (indirectly):

More bullets → the centroid is an average over more directions. Diverse bullets can pull the average toward something broader / more “middle” (sometimes described as washing out sharp specialties). Many bullets saying similar things pull the centroid strongly toward that theme (especially if weights are similar).
Fewer bullets → the centroid is dominated by those few directions; it can look more specific or noisier as a summary of the whole career, depending on what those three bullets say.
So: 10 vs 3 does not change the mathematical type (always one 384‑dim vector per resume). It changes how that vector is formed and therefore where it sits in space—and that in turn affects the cosine score when you compare A to B, but there is no separate “penalty” for 10 vs 3 in the centroid formula itself; the effect shows up through content and diversity of those bullets (and your recency weights).

**CentroidSim** compares each resume’s **single centroid vector** **C_R**: a **weighted mean** of **Experience** row embeddings from `corpus_with_embedding.csv` (same dimension as one bullet, e.g. 384). **CentroidSim** vs `TARGET_RESUME_ID` is **cosine similarity** between **C_target** and each **C_R** via `sklearn.metrics.pairwise.cosine_similarity`.

**Configurable (top of the next code cell):** three weights `WEIGHT_EXPERIENCE_RECENT` / `MID` / `OLD`, and two month cutoffs `RECENCY_MONTHS_AGO_MAX_RECENT` and `RECENCY_MONTHS_AGO_MAX_MID` from `RecencyMonthsAgo` (**NaN** → recent tier). Invalid thresholds (`MAX_RECENT >= MAX_MID`) raise **`ValueError`**.

A centroid captures **broad semantic overlap** across bullets, but **averaging can wash out** sharp specialties; **many similar bullets** pull **C_R** toward that theme. Use this as **component 2** of a future composite, not a full similarity verdict.

**Checks:** target self-similarity **≈ 1**; all cosines in **[-1, 1]**. Spot-read Experience text for a few pairs against the ranking.

In [5]:
from sklearn.metrics.pairwise import cosine_similarity

# --- tune experience row weights (RecencyMonthsAgo → α) ---
WEIGHT_EXPERIENCE_RECENT = 1.25
WEIGHT_EXPERIENCE_MID = 1.0
WEIGHT_EXPERIENCE_OLD = 0.75
RECENCY_MONTHS_AGO_MAX_RECENT = 12
RECENCY_MONTHS_AGO_MAX_MID = 48


def recency_weight(
    months_ago: float,
    w_recent: float,
    w_mid: float,
    w_old: float,
    max_recent: float,
    max_mid: float,
) -> float:
    if max_recent >= max_mid:
        raise ValueError(
            f"RECENCY_MONTHS_AGO_MAX_RECENT ({max_recent}) must be < "
            f"RECENCY_MONTHS_AGO_MAX_MID ({max_mid})"
        )
    if pd.isna(months_ago) or float(months_ago) <= max_recent:
        return w_recent
    if float(months_ago) <= max_mid:
        return w_mid
    return w_old


if not OUT_EMB_PATH.is_file():
    raise FileNotFoundError(f"Embedded corpus not found: {OUT_EMB_PATH.resolve()}")
if not META_PATH.is_file():
    raise FileNotFoundError(f"Resume metadata not found: {META_PATH.resolve()}")

emb_df2 = pd.read_csv(OUT_EMB_PATH, dtype={"ResumeID": int, "ItemID": int})
exp_mask = emb_df2["Section"].astype(str).str.strip() == "Experience"
exp_df = emb_df2.loc[exp_mask & emb_df2["embedding"].astype(str).str.len().gt(0)].copy()

all_resume_ids = sorted(emb_df2["ResumeID"].unique())
exp_counts = exp_df.groupby("ResumeID").size()
missing_exp = [rid for rid in all_resume_ids if rid not in exp_counts.index or exp_counts.loc[rid] == 0]
if missing_exp:
    raise ValueError(f"No non-empty Experience embedding for ResumeID(s): {missing_exp}")

if exp_df["RecencyMonthsAgo"].isna().mean() > 0.5:
    print("Warning: over half of Experience rows have NaN RecencyMonthsAgo; those rows use the RECENT weight tier.")

centroids: dict[int, np.ndarray] = {}
for rid in all_resume_ids:
    rows = exp_df.loc[exp_df["ResumeID"] == rid]
    w_list = []
    v_list = []
    for _, row in rows.iterrows():
        alpha = recency_weight(
            row["RecencyMonthsAgo"],
            WEIGHT_EXPERIENCE_RECENT,
            WEIGHT_EXPERIENCE_MID,
            WEIGHT_EXPERIENCE_OLD,
            RECENCY_MONTHS_AGO_MAX_RECENT,
            RECENCY_MONTHS_AGO_MAX_MID,
        )
        v = np.array(json.loads(row["embedding"]), dtype=np.float32)
        w_list.append(alpha)
        v_list.append(v * alpha)
    w_sum = float(np.sum(w_list))
    centroids[rid] = np.sum(np.stack(v_list, axis=0), axis=0) / w_sum

if TARGET_RESUME_ID not in centroids:
    raise ValueError(f"TARGET_RESUME_ID={TARGET_RESUME_ID} has no experience centroid")

ids_c = sorted(centroids.keys())
Xc = np.stack([centroids[rid] for rid in ids_c], axis=0)
target_c = centroids[TARGET_RESUME_ID].reshape(1, -1)
cent_sims = cosine_similarity(target_c, Xc)[0]

centroid_rank_df = (
    pd.DataFrame({"ResumeID": ids_c, "centroid_cosine_similarity": cent_sims.astype(float)})
    .sort_values("centroid_cosine_similarity", ascending=False)
    .reset_index(drop=True)
)
meta_c = pd.read_csv(META_PATH)
centroid_rank_df = centroid_rank_df.merge(meta_c, on="ResumeID", how="left")

ti = ids_c.index(TARGET_RESUME_ID)
self_c = float(cent_sims[ti])
assert abs(self_c - 1.0) < 1e-5, f"expected target centroid self-similarity ~1, got {self_c}"
assert cent_sims.min() >= -1.0001 and cent_sims.max() <= 1.0001

centroid_rank_df

,ResumeID,centroid_cosine_similarity,primary_domain,role_family,seniority_band
0,1,1.000000,data_analytics,healthcare_analytics,mid
1,4,0.697838,data_analytics,healthcare_reporting,early
2,7,0.580856,data_science,research_ds,senior
3,9,0.579455,data_science,forecasting,mid
4,2,0.573389,data_analytics,bi_reporting,mid
5,5,0.555207,data_analytics,product_analytics,mid
6,14,0.544723,finance,fpna,senior
7,3,0.529107,data_analytics,operations_analytics,senior
8,6,0.525987,data_science,ml_engineering,mid
9,8,0.515832,data_science,nlp_search,mid
